# Qwen3.5-9B QLoRA: scaling comparison (r = 48)

Measure how LoRA scaling affects rubric-based grading while keeping rank fixed at **r = 48**.
Train **α=24, α=48, α=96, α=192**, corresponding to **α/r = 0.5, 1, 2, 4**, and compare them with the unadapted **Baseline**.

- Share one 80/10/10 split by unique question, prompt, tokenizer, and completion masks across all configurations.
- Reset the training seed before loading each fresh NF4 base model and adapter; vary only alpha.
- Select each adapter by validation loss, with approximately four validation/checkpoint steps per epoch, plateau LR reduction, and early stopping.
- Disable thinking for training and use greedy decoding for predictions.

Run the setup and data-preparation cells first. The following sections define helpers; training starts in **Run all four scaling configurations**.
A full run requires one CUDA GPU and saves four adapters, predictions, metrics, and reproducibility metadata in a new experiment directory.
The final comparison highlights each metric's winners and uses QWK as the primary metric for the overall configuration ranking.


## Install dependencies

- Use Python 3.12 in the Linux CUDA training environment. Run this setup in a fresh kernel before **Imports and hardware**. If you restart the kernel, rerun setup to restore its CUDA/compiler environment settings.
- Install Python packages and the CUDA 13.0 compiler/headers with pip. The optional DeltaNet kernels may require compilation; failed optional installs leave the PyTorch fallback available.
- Install system GCC/G++ separately if needed, for example with `sudo apt-get install build-essential` on Ubuntu.
- The compilation target defaults to RTX 4090 (`TORCH_CUDA_ARCH_LIST=8.9`); set this environment variable before setup when using a different GPU.
- For reproducing an earlier run, use its recorded package versions rather than upgrading the environment with this cell.
- References: [Qwen3.5](https://huggingface.co/docs/transformers/model_doc/qwen3_5), [TRL SFTTrainer](https://huggingface.co/docs/trl/sft_trainer).


In [ ]:
%pip install ipywidgets
%pip install -U torch transformers accelerate peft datasets bitsandbytes trl sentencepiece pandas scikit-learn ninja packaging huggingface_hub python-dotenv setuptools wheel "cuda-toolkit[nvcc,cudart,cccl,crt,nvvm]>=13.0,<13.1"

import os
import shutil
import subprocess
import sys
from importlib.metadata import distribution
from pathlib import Path

# Locate the CUDA compiler installed by the pip toolkit package.
nvcc_dist = distribution("nvidia-cuda-nvcc")
nvcc = next(
    (Path(nvcc_dist.locate_file(file)).resolve() for file in nvcc_dist.files or []
     if str(file).endswith("/bin/nvcc")),
    None,
)
if nvcc is None or not nvcc.is_file():
    raise RuntimeError("CUDA nvcc was not found; check the CUDA pip installation above.")
cuda_home = nvcc.parent.parent
os.environ["CUDA_HOME"] = str(cuda_home)
os.environ["PATH"] = str(nvcc.parent) + os.pathsep + os.environ.get("PATH", "")
os.environ["CPATH"] = str(cuda_home / "include") + os.pathsep + os.environ.get("CPATH", "")
os.environ.setdefault("TORCH_CUDA_ARCH_LIST", "8.9")  # Default compilation target: RTX 4090.
os.environ.setdefault("MAX_JOBS", "1")
if (system_cc := shutil.which("gcc")):
    os.environ["CC"] = system_cc
if (system_cxx := shutil.which("g++")):
    os.environ["CXX"] = system_cxx
if not system_cc or not system_cxx:
    print("System GCC/G++ missing; on Ubuntu run: sudo apt-get install build-essential")

for package_args in (["flash-linear-attention[cuda]"], ["causal-conv1d", "--no-build-isolation"]):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-U", *package_args], check=False,
    )
    if result.returncode:
        print(f"Optional install failed: {package_args[0]}; using the PyTorch fallback.")


## Imports and hardware

- Check the active Python environment, CUDA availability, compiler, and optional kernels.
- Model loading and training require one CUDA GPU. Set `RUN_TRAINING=False` to stop after data/tokenizer checks; these checks can run on CPU with the required packages installed.
- Load optional `.env` settings; Hugging Face libraries use `HF_TOKEN` from the environment when supplied.


In [ ]:
import ast
import csv
import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

# Set the compiler before importing PyTorch/Triton.
cc = os.environ.get("CC") or next(
    (path for name in ("gcc", "clang") if (path := shutil.which(name))),
    None,
)
if cc:
    os.environ["CC"] = cc

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from dotenv import load_dotenv
from huggingface_hub import login, upload_folder
from IPython.display import display
from peft import LoraConfig, get_peft_model
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

load_dotenv()

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Compiler:", cc)
for package in ("causal_conv1d", "fla"):
    print(f"{package} installed:", importlib.util.find_spec(package) is not None)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory / 1024**3:.1f} GiB")


## Configuration

- Use the repository rubric dataset, located from the notebook directory or repository root. Set `DATASET_PATH` explicitly for another file.
- `SEED=42` controls training; `SPLIT_SEED=42` controls the shared question split.
- `MODEL_REVISION=None` uses the current model revision. Set it to a recorded commit hash to reproduce an earlier run.
- `MAX_SEQ_LENGTH=None` uses the longest formatted training/validation example, rounded up to 32 tokens. An explicit cap rejects overlength examples.
- Physical batch size 1 × accumulation 8 gives effective batch size 8 on one GPU.
- `RUN_TRAINING=True` runs all four configurations. `RUN_TEST_EVALUATION=True` additionally evaluates one baseline and all four adapters on the shared test set.
- The generation budget is `max_new_tokens=max(128, 8 × criterion count)`; it caps output length and does not require a minimum completion length.
- Upload is disabled by default. To publish one evaluated adapter, set `RUN_HF_UPLOAD=True`, `UPLOAD_ALPHA`, and `HF_REPO_ID` explicitly.


In [ ]:
MODEL_ID = "Qwen/Qwen3.5-9B"
MODEL_REVISION = None  # Set to a recorded commit hash to reproduce an earlier run.
DATASET_PATH = next(
    (str(parent / "Datasets/training_data_moh_eng_rubrics.csv")
     for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "Datasets/training_data_moh_eng_rubrics.csv").is_file()),
    "./Datasets/training_data_moh_eng_rubrics.csv",
)
OUTPUT_DIR = "./qwen35-9b-grading-lora-scaling"
RUN_TRAINING = True
RUN_TEST_EVALUATION = True
RUN_HF_UPLOAD = False  # Opt in only when ready to publish one evaluated adapter.
UPLOAD_ALPHA = 96
HF_REPO_ID = "SmuFypTeam5/GradingQlora"
evaluation_completed = False
SEED = 42
SPLIT_SEED = 42
MAX_SEQ_LENGTH = None  # Auto-size from formatted training/validation data.

LORA_R = 48
LORA_ALPHAS = (24, 48, 96, 192)
if LORA_R != 48 or LORA_ALPHAS != (24, 48, 96, 192):
    raise ValueError("This experiment requires r=48 and alpha values 24, 48, 96, 192.")
LORA_DROPOUT = 0.05
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.05
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8  # Effective batch size 8 on one GPU.
VALIDATIONS_PER_EPOCH = 4
LR_REDUCTION_FACTOR = 0.5
LR_PATIENCE = 1
MIN_LEARNING_RATE = 1e-6
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_THRESHOLD = 0.0
GENERATION_MIN_TOKEN_BUDGET = 128
GENERATION_TOKENS_PER_CRITERION = 8

set_seed(SEED)


## Load the dataset

- Require prompt fields, binary labels, criterion maxima, and final-score metadata.
- Keep blank student answers empty and preserve literal `NA` text.


In [ ]:
required_columns = [
    "question", "desired_answer", "student_answer", "criteria[]",
    "criteria_score[]", "criteria_max_score[]", "student_score",
]
if not Path(DATASET_PATH).is_file():
    raise FileNotFoundError(f"Rubric dataset not found: {DATASET_PATH}. Set DATASET_PATH in Configuration.")
df = pd.read_csv(DATASET_PATH, keep_default_na=False)
missing = sorted(set(required_columns) - set(df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if df.empty:
    raise ValueError("The dataset is empty.")
print("Rows:", len(df))
display(df.head())


## Parse and validate rows

- Accept JSON/Python arrays and legacy quoted bracket arrays.
- Use label counts and `Answer` boundaries to preserve commas inside legacy criteria.
- Require nonempty criteria, aligned arrays, binary labels, and finite nonnegative score metadata.
- Keep recorded final scores even if they differ from the weighted criterion totals.


In [ ]:
def parse_array(value, expected_count=None):
    """Parse structured arrays or legacy bracket text without losing criterion commas."""
    items = value if isinstance(value, list) else None
    if items is None:
        if not isinstance(value, str):
            raise TypeError(f"Expected an array or string, got {type(value).__name__}.")
        text = value.strip()
        # Unwrap quoted JSON/Python strings before handling legacy unquoted entries.
        for _ in range(3):
            try:
                parsed = json.loads(text)
            except json.JSONDecodeError:
                try:
                    parsed = ast.literal_eval(text)
                except (ValueError, SyntaxError):
                    break
            if isinstance(parsed, list):
                items = parsed
                break
            if not isinstance(parsed, str):
                raise ValueError(f"Expected a list, got {value!r}.")
            text = parsed.strip()
        if items is None:
            if not (text.startswith("[") and text.endswith("]")):
                raise ValueError(f"Expected a bracketed list, got {value!r}.")
            inner = text[1:-1].strip()
            if not inner:
                items = []
            elif expected_count == 1:
                items = [inner]
            elif re.search(r",\s*(?=Answer\s)", inner):
                items = re.split(r",\s*(?=Answer\s)", inner)
            else:
                items = next(csv.reader([inner], skipinitialspace=True))
            items = [item.strip() for item in items]
    if expected_count is not None and len(items) != expected_count:
        raise ValueError(f"Expected {expected_count} entries, parsed {len(items)}: {value!r}")
    return items


def parse_binary_scores(value):
    values = parse_array(value)
    if any(type(x) not in (int, str) or str(x) not in ("0", "1") for x in values):
        raise ValueError(f"Expected binary integers, got {value!r}")
    return [int(x) for x in values]


def parse_score_metadata(value):
    values = parse_array(value)
    if any(isinstance(x, bool) for x in values):
        raise ValueError("Criterion maxima cannot be booleans.")
    return [float(x) for x in values]


records = []
errors = []
for index, row in enumerate(df.to_dict("records")):
    try:
        scores = parse_binary_scores(row["criteria_score[]"])
        criteria = parse_array(row["criteria[]"], expected_count=len(scores))
        maxima = parse_score_metadata(row["criteria_max_score[]"])
        final_score = float(row["student_score"])
        if not criteria or any(not isinstance(c, str) or not c.strip() for c in criteria):
            raise ValueError("Criteria must be nonempty strings.")
        if len(criteria) != len(maxima):
            raise ValueError("Criterion, label, and maximum-score counts must match.")
        if any(not math.isfinite(x) or x < 0 for x in [*maxima, final_score]):
            raise ValueError("Score metadata must be finite and nonnegative.")
        if not str(row["question"]).strip() or not str(row["desired_answer"]).strip():
            raise ValueError("Question and desired answer must be nonempty.")
        records.append({
            "row_index": index,
            **{key: str(row[key]) for key in ("question", "desired_answer", "student_answer")},
            "criteria": criteria, "criteria_score": scores,
            "criteria_max_score": maxima, "student_score": final_score,
        })
    except (ValueError, TypeError, SyntaxError) as error:
        errors.append(f"Row {index}: {error}")
if errors:
    raise ValueError(f"{len(errors)} invalid rows:\n" + "\n".join(errors[:20]))
print(f"All {len(records)} rows passed validation.")
score_disagreements = sum(
    not math.isclose(
        sum(score * maximum for score, maximum in zip(row["criteria_score"], row["criteria_max_score"])),
        row["student_score"],
    )
    for row in records
)
print("Recorded final scores differing from criterion totals:", score_disagreements)


## Grading prompt

- Return only the ordered JSON array of binary decisions.
- Share one prompt builder between training and inference.


In [ ]:
SYSTEM_PROMPT = (
    "You are a strict grading assistant.\n\n"
    "Your task is to evaluate whether a student's answer fulfills each grading criterion.\n\n"
    "For each criterion:\n"
    "- Output 1 if the criterion is fulfilled.\n"
    "- Output 0 if the criterion is not fulfilled.\n\n"
    "Evaluate each criterion independently.\n\n"
    "Return ONLY a valid JSON array containing 0 or 1.\n"
    "The output array must contain exactly one value for each criterion, "
    "in the same order as the criteria provided.\n\n"
    "Do not include explanations.\n"
    "Do not include markdown.\n"
    "Do not include any text before or after the JSON array."
)

print(SYSTEM_PROMPT)


## Build examples

- Retain structured fields for evaluation; do not parse fields back out of prompt text.
- Keep the assistant completion separate from the prompt.


In [ ]:
def make_prompt(question, desired_answer, student_answer, criteria):
    """Build the shared grading prompt without score metadata."""
    criteria_text = "\n".join(f"{i}. {text}" for i, text in enumerate(criteria, 1))
    user_text = (
        f"Question:\n{question}\n\n"
        f"Desired answer:\n{desired_answer}\n\n"
        f"Student answer:\n{student_answer}\n\n"
        f"Criteria:\n{criteria_text}"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]


dataset = Dataset.from_list([
    {
        **row,
        "prompt": make_prompt(row["question"], row["desired_answer"], row["student_answer"], row["criteria"]),
        "completion": [{"role": "assistant", "content": json.dumps(row["criteria_score"])}],
    }
    for row in records
])
print(dataset[0]["prompt"])


## Split by question

- Shuffle unique stripped question strings with `SPLIT_SEED`.
- Allocate 80/10/10% of questions; row proportions can differ.
- Require nonempty, disjoint train, validation, and test sets.


In [ ]:
question_keys = [row["question"].strip() for row in records]
unique_questions = list(dict.fromkeys(question_keys))
random.Random(SPLIT_SEED).shuffle(unique_questions)
n_questions = len(unique_questions)
train_end = int(n_questions * 0.80)
validation_end = train_end + int(n_questions * 0.10)
question_splits = {
    "train": set(unique_questions[:train_end]),
    "validation": set(unique_questions[train_end:validation_end]),
    "test": set(unique_questions[validation_end:]),
}
if any(not questions for questions in question_splits.values()):
    raise ValueError("Too few unique questions for nonempty 80/10/10 splits.")
assert sum(map(len, question_splits.values())) == len(set.union(*question_splits.values()))
question_to_split = {q: name for name, questions in question_splits.items() for q in questions}
split_indices = {name: [] for name in question_splits}
for index, question in enumerate(question_keys):
    split_indices[question_to_split[question]].append(index)
splits = {name: dataset.select(indices) for name, indices in split_indices.items()}
train_dataset, validation_dataset, test_dataset = (splits[name] for name in ("train", "validation", "test"))
for name, split in splits.items():
    print(f"{name}: {len(split)} rows / {len(question_splits[name])} questions")


## Tokenizer

- Share one tokenizer and chat template for training, validation, and test generation, with thinking disabled.
- Use the tokenizer's resolved commit hash for every model load when available; record the revision in `experiment.json`.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
# Prefer the resolved commit hash when the tokenizer provides it.
MODEL_REVISION = tokenizer.init_kwargs.get("_commit_hash") or MODEL_REVISION
print("Tokenizer:", tokenizer.__class__.__name__)


## Tokenize and verify completion masks

- Pre-tokenize training/validation examples; pass only token IDs and labels to TRL.
- Verify the generation prompt is an exact prefix of the full conversation.
- Mask prompt labels with `-100`; supervise the complete assistant turn, including its end token.
- Measure actual lengths before model loading; never silently truncate completions.


In [ ]:
def tokenize_example(example):
    """Tokenize one conversation and mask its prompt labels."""
    prompt_ids = tokenizer.apply_chat_template(
        example["prompt"], tokenize=True, return_dict=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    input_ids = tokenizer.apply_chat_template(
        example["prompt"] + example["completion"], tokenize=True, return_dict=False,
        add_generation_prompt=False, enable_thinking=False,
    )
    boundary = len(prompt_ids)
    if input_ids[:boundary] != prompt_ids or boundary >= len(input_ids):
        raise ValueError(f"Invalid prompt/completion boundary at row {example['row_index']}.")
    return {"input_ids": input_ids, "labels": [-100] * boundary + input_ids[boundary:]}


formatted_train, formatted_validation = (
    split.map(tokenize_example, remove_columns=split.column_names, desc=f"Tokenize {name}")
    for name, split in (("train", train_dataset), ("validation", validation_dataset))
)
lengths = [len(ids) for split in (formatted_train, formatted_validation) for ids in split["input_ids"]]
measured_max_length = max(lengths)
sequence_limit = MAX_SEQ_LENGTH if MAX_SEQ_LENGTH is not None else math.ceil(measured_max_length / 32) * 32
if sequence_limit < measured_max_length:
    raise ValueError(
        f"{sum(length > sequence_limit for length in lengths)} examples exceed {sequence_limit} tokens; "
        f"longest is {measured_max_length}. Increase MAX_SEQ_LENGTH or use None."
    )
print("Training/validation token lengths:", pd.Series(lengths).describe().round(1).to_dict())
print("Sequence limit:", sequence_limit)
print("Example supervised completion:", tokenizer.decode(
    [label for label in formatted_train[0]["labels"] if label != -100], skip_special_tokens=False,
))


## Quantisation and model loading

- Load a fresh text backbone per configuration with the same NF4, double quantisation, and BF16/FP16 compute.
- SDPA handles full attention; optional DeltaNet kernels handle linear attention.
- Keep one model on one GPU and release it before loading the next configuration.


In [ ]:
def load_quantized_base_model():
    """Load a fresh NF4 text backbone on the single training GPU."""
    if not torch.cuda.is_available():
        raise RuntimeError("QLoRA training requires a CUDA GPU.")
    if not cc:
        raise RuntimeError("Install a C compiler with the dependency cell, then restart the kernel.")
    if int(os.environ.get("WORLD_SIZE", "1")) != 1:
        raise RuntimeError("This notebook's batch and validation schedule assume one training GPU.")
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, quantization_config=bnb_config, device_map={"": 0},
        dtype=compute_dtype, attn_implementation="sdpa",
    )
    model.config.use_cache = False
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    print("Loaded in 4-bit:", model.is_loaded_in_4bit, "| Compute dtype:", compute_dtype)
    return model, compute_dtype


## Attach LoRA adapters

- Freeze the base weights and create a fresh rank-48 adapter for the current alpha.
- Target the text backbone's attention and MLP projections: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, and `down_proj` where present.
- Keep dropout at 0.05, disable bias training, and use ordinary α/r scaling (`use_rslora=False`).


In [ ]:
def attach_fresh_lora(model, alpha):
    """Freeze the base model and attach a new rank-48 adapter."""
    for param in model.parameters():
        param.requires_grad_(False)
    candidate_suffixes = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    target_modules = [
        name for name, module in model.named_modules()
        if name.startswith("model.layers.") and isinstance(module, torch.nn.Linear)
        and name.rsplit(".", 1)[-1] in candidate_suffixes
    ]
    if not target_modules:
        raise RuntimeError("No text projection targets found; inspect model.named_modules().")
    print("LoRA projection layers:", len(target_modules))
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=alpha, lora_dropout=LORA_DROPOUT,
        use_rslora=False, bias="none", task_type="CAUSAL_LM", target_modules=target_modules,
    ))
    model.print_trainable_parameters()
    return model


## Training configuration

- Reuse the same training arguments for every alpha, with separate output directories.
- Use paged 8-bit AdamW, mixed precision, and non-reentrant gradient checkpointing.
- Compute NLL on completion tokens only. Keep dynamic padding enabled and activation offloading, packing, and padding-free mode disabled.
- Evaluate and save approximately every quarter epoch, rounded to an optimizer step.
- Reduce LR after two consecutive validation checks without sufficient improvement (`LR_PATIENCE=1`), and stop after four unimproved checks.
- Restore the lowest-validation-loss checkpoint before saving and test generation. Test metrics do not influence checkpoint selection.


In [ ]:
microbatches_per_epoch = math.ceil(len(formatted_train) / TRAIN_BATCH_SIZE)
optimizer_steps_per_epoch = math.ceil(microbatches_per_epoch / GRADIENT_ACCUMULATION_STEPS)
quarter_epoch_steps = max(1, round(optimizer_steps_per_epoch / VALIDATIONS_PER_EPOCH))
print("Optimizer steps per epoch:", optimizer_steps_per_epoch)
print("Evaluate/save every:", quarter_epoch_steps, "optimizer steps")


def make_training_args(run_dir, compute_dtype):
    """Build shared training settings with a configuration-specific output path."""
    return SFTConfig(
        output_dir=str(run_dir),
        seed=SEED,
        data_seed=SEED,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        optim="paged_adamw_8bit",
        lr_scheduler_type="reduce_lr_on_plateau",
        lr_scheduler_kwargs={
            "mode": "min", "factor": LR_REDUCTION_FACTOR,
            "patience": LR_PATIENCE, "min_lr": MIN_LEARNING_RATE,
        },
        bf16=(compute_dtype == torch.bfloat16),
        fp16=(compute_dtype == torch.float16),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        activation_offloading=False,
        loss_type="nll",
        completion_only_loss=True,
        max_length=sequence_limit,
        packing=False,
        padding_free=False,
        logging_steps=max(1, quarter_epoch_steps // 2),
        save_strategy="steps",
        save_steps=quarter_epoch_steps,
        eval_strategy="steps",
        eval_steps=quarter_epoch_steps,
        prediction_loss_only=True,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=3,
        report_to="none",
    )


## Create the trainer

- Create a new trainer and early-stopping callback for each fresh adapter.
- Reuse the prepared training/validation examples and tokenizer; validation computes loss only.


In [ ]:
def create_trainer(model, run_dir, compute_dtype):
    """Create independent training state for the current adapter."""
    return SFTTrainer(
        model=model, args=make_training_args(run_dir, compute_dtype),
        train_dataset=formatted_train, eval_dataset=formatted_validation,
        processing_class=tokenizer,
        callbacks=[EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )],
    )


## Check the collated loss mask

- Before training each configuration, inspect up to two examples using its trainer's data collator.
- Prompt and padding labels must be `-100`; completion tokens must remain supervised.


In [ ]:
def verify_completion_masks(trainer):
    """Verify that the collator supervises completion tokens only."""
    samples = [trainer.train_dataset[i] for i in range(min(2, len(trainer.train_dataset)))]
    batch = trainer.data_collator(samples)
    for index, sample in enumerate(samples):
        labels = batch["labels"][index]
        length = len(sample["labels"])
        assert labels[:length].tolist() == sample["labels"], "Collator changed the completion mask."
        assert (labels[length:] == -100).all(), "Padding contributes to loss."
        assert (labels[:length] == -100).any() and (labels[:length] != -100).any()
    print("Prompt/completion/padding masks verified.")


## Train and save

- Train the current adapter, restore its lowest-validation-loss checkpoint, and save the adapter and tokenizer in `alpha_<value>/`.
- Save training metrics, loss history, and best-checkpoint metadata alongside the adapter.
- Release optimizer, scheduler, and gradient storage before inference, including cleanup after a failed training run.


In [ ]:
def train_and_save(model, run_dir, compute_dtype):
    """Save the validation-selected adapter and release training storage."""
    trainer = None
    try:
        trainer = create_trainer(model, run_dir, compute_dtype)
        verify_completion_masks(trainer)
        train_result = trainer.train()
        trainer.save_model(str(run_dir))
        tokenizer.save_pretrained(str(run_dir))
        (run_dir / "training_metrics.json").write_text(json.dumps(train_result.metrics, indent=2) + "\n")
        pd.DataFrame(trainer.state.log_history).to_csv(run_dir / "training_history.csv", index=False)
        (run_dir / "best_checkpoint.json").write_text(json.dumps({
            "checkpoint": trainer.state.best_model_checkpoint,
            "validation_loss": trainer.state.best_metric,
        }, indent=2) + "\n")
        print("Saved best adapter and tokenizer to:", run_dir)
        display(train_result.metrics)
        return trainer.model
    finally:
        if trainer is not None:
            trainer.model.zero_grad(set_to_none=True)
            trainer.optimizer = None
            trainer.lr_scheduler = None
            trainer.accelerator.free_memory()
        trainer = None
        gc.collect()
        torch.cuda.empty_cache()


## Prepare inference

- Reuse the saved adapter's model after training storage has been released.
- Disable gradient checkpointing, enable the KV cache, and switch to evaluation mode.


In [ ]:
def prepare_inference(model):
    """Switch the trained model to cached inference."""
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    model.eval()


## Prediction helper

- Use the shared prompt, greedy decoding, and no thinking.
- Accept only integer `0`/`1` arrays of the correct length; reject floats and booleans.
- If JSON decoding fails, recover the last binary array of the expected length from surrounding text and report it as `recovered`.


In [ ]:
def parse_prediction(raw_output, expected_count):
    """Validate binary decisions, recovering an embedded array when needed."""
    recovered = False
    try:
        scores = json.loads(raw_output)
    except json.JSONDecodeError:
        candidates = re.findall(r"\[\s*[01](?:\s*,\s*[01])*\s*\]", raw_output)
        scores = next((
            parsed for candidate in reversed(candidates)
            if len(parsed := json.loads(candidate)) == expected_count
        ), None)
        recovered = scores is not None
    error = None
    if not isinstance(scores, list):
        error = "Output was not a JSON array."
    elif len(scores) != expected_count:
        error = f"Expected {expected_count} scores, received {len(scores)}."
    elif any(type(score) is not int or score not in (0, 1) for score in scores):
        error = "Scores must be integer 0 or 1."
    return {
        "valid": error is None, "raw_output": raw_output,
        "scores": scores if error is None else None,
        "error": error, "recovered": recovered,
    }


@torch.inference_mode()
def predict_criteria_scores(question, desired_answer, student_answer, criteria, prediction_model):
    """Generate one greedy rubric prediction with the shared parser."""
    inputs = tokenizer.apply_chat_template(
        make_prompt(question, desired_answer, student_answer, criteria),
        tokenize=True, return_dict=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False,
    ).to(prediction_model.get_input_embeddings().weight.device)
    generated = prediction_model.generate(
        **inputs, do_sample=False,
        max_new_tokens=max(GENERATION_MIN_TOKEN_BUDGET, len(criteria) * GENERATION_TOKENS_PER_CRITERION),
    )
    answer_ids = generated[0, inputs["input_ids"].shape[-1]:]
    raw_output = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
    return parse_prediction(raw_output, len(criteria))


## Manual prediction

- Run the same illustrative example after each adapter finishes training, including when test evaluation is disabled.
- The expected rubric array is `[1, 0, 0]`. This example is separate from the held-out metrics.


In [ ]:
def show_manual_prediction(model):
    """Print an illustrative prediction outside the test-set metrics."""
    result = predict_criteria_scores(
        question="Explain why HTTPS is more secure than HTTP.",
        desired_answer="HTTPS encrypts network traffic using TLS.",
        student_answer="HTTPS protects transmitted data using encryption.",
        criteria=["Mentions encryption", "Mentions TLS", "Claims HTTPS sends plaintext"],
        prediction_model=model,
    )
    print(result)


## Evaluate the held-out test set

- Use one evaluator, test split, prompt, parser, and decoding procedure for the baseline and all four adapters.
- Compute the final score as the sum of each predicted binary decision × its criterion maximum, compared with the recorded `student_score`.
- Invalid outputs count as failures for exact-array and final-score accuracy; criterion metrics, MAE, RMSE, Pearson, and QWK use valid outputs only.
- Pearson is undefined for fewer than two valid predictions or constant true/predicted scores. Other undefined metrics remain `NaN`.
- Save every metric, including invalid/recovered-output rates, plus raw predictions and per-example errors.


In [ ]:
def evaluate_current_model(dataset, model_name, prediction_model):
    """Score one model state with the shared invalid-output and final-score rules."""
    criterion_true = []
    criterion_pred = []
    exact_correct = 0
    invalid_count = 0
    final_score_errors = []
    final_score_exact = 0
    final_score_within_one = 0
    valid_true_final_scores = []
    valid_predicted_final_scores = []
    rows = []

    for example in dataset:
        true_scores = example["criteria_score"]
        max_scores = example["criteria_max_score"]
        true_final_score = example["student_score"]
        prediction = predict_criteria_scores(
            example["question"], example["desired_answer"],
            example["student_answer"], example["criteria"], prediction_model=prediction_model,
        )

        # Invalid arrays are failures for response-level exact matching.
        is_valid = prediction["valid"]
        pred_scores = prediction["scores"] if is_valid else None
        is_exact = bool(is_valid and pred_scores == true_scores)
        exact_correct += int(is_exact)
        invalid_count += int(not is_valid)

        # Criterion metrics require aligned arrays, so invalid arrays are omitted.
        if is_valid:
            criterion_true.extend(true_scores)
            criterion_pred.extend(pred_scores)

            # Convert binary criterion decisions into the final numeric score.
            predicted_final_score = sum(
                score * maximum for score, maximum in zip(pred_scores, max_scores)
            )
            score_error = abs(predicted_final_score - true_final_score)
            final_score_errors.append(score_error)
            final_score_exact += int(score_error == 0)
            final_score_within_one += int(score_error <= 1)
            valid_true_final_scores.append(true_final_score)
            valid_predicted_final_scores.append(predicted_final_score)
        else:
            predicted_final_score = None
            score_error = None

        rows.append({
            "model": model_name,
            "row_index": example["row_index"],
            "raw_output": prediction["raw_output"],
            "recovered": prediction["recovered"],
            "error": prediction["error"],
            "ground_truth": true_scores,
            "prediction": pred_scores if is_valid else prediction["raw_output"],
            "valid": is_valid,
            "exact_match": is_exact,
            "true_final_score": true_final_score,
            "predicted_final_score": predicted_final_score,
            "final_score_absolute_error": score_error,
        })

    total = len(dataset)
    precision, recall, f1, _ = (
        precision_recall_fscore_support(criterion_true, criterion_pred, average="binary", zero_division=0)
        if criterion_true else (float("nan"),) * 4
    )
    if (
        len(valid_true_final_scores) > 1
        and len(set(valid_true_final_scores)) > 1
        and len(set(valid_predicted_final_scores)) > 1
    ):
        final_score_pearson = float(
            np.corrcoef(valid_true_final_scores, valid_predicted_final_scores)[0, 1]
        )
    else:
        final_score_pearson = float("nan")
    if valid_true_final_scores:
        final_score_qwk = float(cohen_kappa_score(
            valid_true_final_scores, valid_predicted_final_scores, weights="quadratic"
        ))
    else:
        final_score_qwk = float("nan")

    metrics = {
        "model": model_name,
        "examples": total,
        "valid_examples": total - invalid_count,
        "recovered_output_rate": sum(row["recovered"] for row in rows) / total if total else 0.0,
        "invalid_output_rate": invalid_count / total if total else 0.0,
        "exact_match_accuracy": exact_correct / total if total else 0.0,
        "per_criterion_accuracy": accuracy_score(criterion_true, criterion_pred) if criterion_true else float("nan"),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        # Invalid generations count as incorrect for the two score accuracies.
        "final_score_exact_accuracy": final_score_exact / total if total else 0.0,
        "final_score_within_one_accuracy": final_score_within_one / total if total else 0.0,
        # MAE, RMSE, Pearson, and QWK describe valid numeric predictions only.
        "final_score_mae": (
            sum(final_score_errors) / len(final_score_errors)
            if final_score_errors else float("nan")
        ),
        "final_score_rmse": (
            (sum(error ** 2 for error in final_score_errors) / len(final_score_errors)) ** 0.5
            if final_score_errors else float("nan")
        ),
        "final_score_pearson": final_score_pearson,
        "final_score_qwk": final_score_qwk,
    }
    return metrics, rows


## Run all four scaling configurations

- For each alpha, reset `SEED`, load a fresh base model and adapter, train with the shared settings, and run the manual prediction.
- When `RUN_TEST_EVALUATION=True`, evaluate the baseline once with the first trained adapter disabled, then evaluate all four adapters. Frozen base weights make this a reference to the same unadapted model.
- Release each model before loading the next configuration. Each experiment creates a new `run-<UTC timestamp>/` directory under `OUTPUT_DIR`.
- Save `experiment.json` with the dataset checksum, split row indices, question splits, model revision, package versions, hardware, and shared settings.
- Save per-model metric CSVs and prediction JSONL files. Update `scaling_evaluation_comparison.csv` with all original metrics after each configuration completes evaluation.
- The final table requires all five evaluations. With test evaluation disabled, only training artifacts and manual predictions are produced.
- Reproduce a run with its recorded model revision, dataset, package versions, seeds, and hardware; CUDA kernels can still introduce numerical differences across environments.


In [ ]:
def save_evaluation(run_dir, label, metrics, rows):
    """Persist complete metrics and raw predictions for one model state."""
    pd.DataFrame([metrics]).to_csv(run_dir / f"{label}_metrics.csv", index=False)
    pd.DataFrame(rows).to_json(run_dir / f"{label}_predictions.jsonl", orient="records", lines=True)


def run_configuration(alpha, experiment_dir, include_baseline=False):
    """Train one fresh adapter, optionally evaluate it, and release GPU memory."""
    run_dir = experiment_dir / f"alpha_{alpha}"
    run_dir.mkdir(exist_ok=False)
    model = None
    results, predictions = [], []
    try:
        set_seed(SEED)
        model, compute_dtype = load_quantized_base_model()
        model = attach_fresh_lora(model, alpha)
        model = train_and_save(model, run_dir, compute_dtype)
        prepare_inference(model)
        show_manual_prediction(model)
        if RUN_TEST_EVALUATION:
            if include_baseline:
                with model.disable_adapter():
                    metrics, rows = evaluate_current_model(test_dataset, "Baseline", prediction_model=model)
                save_evaluation(experiment_dir, "baseline", metrics, rows)
                results.append(metrics)
                predictions.extend(rows)
            metrics, rows = evaluate_current_model(test_dataset, f"α={alpha}", prediction_model=model)
            save_evaluation(run_dir, "test", metrics, rows)
            results.append(metrics)
            predictions.extend(rows)
        return results, predictions
    finally:
        model = None
        gc.collect()
        torch.cuda.empty_cache()


all_metrics, all_prediction_rows = [], []
experiment_dir = None
evaluation_completed = False
if RUN_TRAINING:
    experiment_dir = Path(OUTPUT_DIR) / datetime.now(timezone.utc).strftime("run-%Y%m%dT%H%M%S-%fZ")
    experiment_dir.mkdir(parents=True, exist_ok=False)
    experiment_manifest = {
        "model": MODEL_ID, "model_revision": MODEL_REVISION,
        "dataset_path": str(Path(DATASET_PATH).resolve()),
        "dataset_sha256": hashlib.sha256(Path(DATASET_PATH).read_bytes()).hexdigest(),
        "seed": SEED, "split_seed": SPLIT_SEED,
        "lora_rank": LORA_R,
        "configurations": [{"alpha": alpha, "scaling": alpha / LORA_R} for alpha in LORA_ALPHAS],
        "split_row_indices": {name: [row["row_index"] for row in split] for name, split in splits.items()},
        "question_splits": {name: sorted(questions) for name, questions in question_splits.items()},
        "sequence_limit": sequence_limit,
        "shared_training_args": make_training_args(
            experiment_dir, torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        ).to_dict(),
        "lora_dropout": LORA_DROPOUT, "use_rslora": False,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
        "generation": {
            "do_sample": False, "enable_thinking": False,
            "max_new_tokens_floor": GENERATION_MIN_TOKEN_BUDGET,
            "tokens_per_criterion": GENERATION_TOKENS_PER_CRITERION,
        },
        "package_versions": {name: version(name) for name in (
            "torch", "transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets", "numpy", "pandas", "scikit-learn",
        )},
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "cuda_version": torch.version.cuda,
    }
    (experiment_dir / "experiment.json").write_text(json.dumps(experiment_manifest, indent=2, default=str) + "\n")
    print("Experiment directory:", experiment_dir)
    for index, alpha in enumerate(LORA_ALPHAS):
        print(f"Starting α={alpha}, r={LORA_R}, scaling={alpha / LORA_R:g} ({index + 1}/{len(LORA_ALPHAS)})")
        metrics, rows = run_configuration(alpha, experiment_dir, include_baseline=(index == 0))
        all_metrics.extend(metrics)
        all_prediction_rows.extend(rows)
        if metrics:
            pd.DataFrame(all_metrics).to_csv(experiment_dir / "scaling_evaluation_comparison.csv", index=False)
    if RUN_TEST_EVALUATION:
        expected_models = ["Baseline", *(f"α={alpha}" for alpha in LORA_ALPHAS)]
        if [metrics["model"] for metrics in all_metrics] != expected_models:
            raise RuntimeError("The baseline and all four configurations must finish evaluation.")
        evaluation_completed = True
else:
    print("Data checks complete. Set RUN_TRAINING=True to run all four scaling configurations.")


## Inspect predictions

- Preview the first 20 predictions retained in memory, beginning with the baseline.
- Keep the model label, raw output, parsed decisions, validity/recovery status, and final-score errors. Complete records are also saved in per-model prediction JSONL files.


In [ ]:
if evaluation_completed:
    final_score_predictions_df = pd.DataFrame(all_prediction_rows)
    display(final_score_predictions_df[[
        "model", "row_index", "ground_truth", "prediction", "raw_output",
        "true_final_score", "predicted_final_score", "final_score_absolute_error", "valid", "recovered",
    ]].head(20))


## Scaling comparison with the baseline

- Display **Baseline, α=24, α=48, α=96, α=192**, with rank fixed at 48.
- **QWK, Pearson, Accuracy:** higher is better. **RMSE, MAE:** lower is better.
- **Accuracy** is exact final-score accuracy over all test examples, counting invalid outputs as failures. The other four metrics use valid predictions only; review the saved validity rates alongside them.
- Highlight all tied metric winners using unrounded values. Display four decimal places and show undefined values as `NaN`.
- Rank adapter configurations by QWK, then Pearson, lower RMSE, lower MAE, and Accuracy to resolve ties. Preserve ties across all five metrics, place undefined secondary metrics last, and exclude configurations with undefined QWK from the overall ranking.
- Print each metric's winners, the highest-ranked adapter configuration(s), and their metric differences from Baseline. Positive differences favor the adapter for QWK/Pearson/Accuracy; negative differences favor it for RMSE/MAE.
- Export this concise table as `scaling_comparison.csv`; `scaling_evaluation_comparison.csv` retains the complete metric schema. The single-seed comparison describes observed performance and does not establish statistical significance.


In [ ]:
METRIC_COLUMNS = {
    "final_score_qwk": "QWK",
    "final_score_pearson": "Pearson",
    "final_score_rmse": "RMSE",
    "final_score_mae": "MAE",
    "final_score_exact_accuracy": "Accuracy",
}
HIGHER_IS_BETTER = ["QWK", "Pearson", "Accuracy"]
LOWER_IS_BETTER = ["RMSE", "MAE"]


def build_comparison(all_metrics):
    """Validate complete results and select the five displayed score metrics."""
    results = pd.DataFrame(all_metrics).set_index("model")
    row_order = ["Baseline", *(f"α={alpha}" for alpha in LORA_ALPHAS)]
    if results.index.has_duplicates or set(results.index) != set(row_order):
        raise ValueError("Require exactly one baseline and one result for each configured alpha.")
    if results["examples"].nunique() != 1 or results["examples"].iloc[0] != len(test_dataset):
        raise ValueError("All five models must be evaluated on the complete shared test set.")
    return results.loc[row_order, list(METRIC_COLUMNS)].rename(columns=METRIC_COLUMNS)


def metric_winners(table):
    """Return all tied winners per metric, omitting undefined values."""
    winners = {}
    for metric in table.columns:
        values = table[metric].dropna()
        best = values.max() if metric in HIGHER_IS_BETTER else values.min()
        winners[metric] = values.index[values.eq(best)].tolist()
    return winners


def overall_configurations(table):
    """Rank adapters by QWK and the documented secondary metrics."""
    # QWK is primary; never substitute a secondary metric for undefined QWK.
    candidates = table.drop(index="Baseline").dropna(subset=["QWK"])
    if candidates.empty:
        return []
    ranked = candidates.sort_values(
        ["QWK", "Pearson", "RMSE", "MAE", "Accuracy"],
        ascending=[False, False, True, True, False], na_position="last", kind="stable",
    )
    best = ranked.iloc[0]
    tied = ranked.eq(best) | (ranked.isna() & best.isna())
    return ranked.index[tied.all(axis=1)].tolist()


if evaluation_completed:
    comparison_df = build_comparison(all_metrics)
    print("LoRA scaling comparison (r = 48); highlighted cells are metric winners.")
    display(
        comparison_df.style.format("{:.4f}", na_rep="NaN")
        .highlight_max(subset=HIGHER_IS_BETTER, color="#d9ead3")
        .highlight_min(subset=LOWER_IS_BETTER, color="#d9ead3")
    )
    comparison_df.to_csv(experiment_dir / "scaling_comparison.csv")
    for metric, winners in metric_winners(comparison_df).items():
        print(f"Best {metric}: {', '.join(winners) if winners else 'undefined for all models'}")
    best_configurations = overall_configurations(comparison_df)
    if best_configurations:
        print("Best configuration(s), QWK first with the documented tie-breaks:", ", ".join(best_configurations))
        for label in best_configurations:
            delta = comparison_df.loc[label] - comparison_df.loc["Baseline"]
            print(f"{label} minus Baseline: " + ", ".join(f"{metric} {value:+.4f}" for metric, value in delta.items()))
    else:
        print("Overall configuration unavailable: all adapter QWK values are undefined.")
    print("Complete metrics, validity rates, and raw predictions saved in:", experiment_dir)


## Optional upload of one evaluated adapter

- Upload is disabled by default. Enable `RUN_HF_UPLOAD` and choose `UPLOAD_ALPHA` and `HF_REPO_ID` in Configuration.
- Require completed training and held-out evaluation for all four configurations and the baseline.
- Upload files from the selected `alpha_<value>/` directory, including its adapter, tokenizer, and evaluation/training artifacts; exclude intermediate `checkpoint-*` folders.
- `login()` requests Hugging Face authentication. The target repository must already exist and the authenticated account must have write access.


In [ ]:
if RUN_HF_UPLOAD:
    if not (RUN_TRAINING and RUN_TEST_EVALUATION):
        raise RuntimeError(
            "Set RUN_TRAINING=True and RUN_TEST_EVALUATION=True before uploading."
        )
    if not evaluation_completed:
        raise RuntimeError("Complete the held-out evaluation cells before uploading.")
    if UPLOAD_ALPHA not in LORA_ALPHAS:
        raise ValueError("UPLOAD_ALPHA must be one of LORA_ALPHAS.")
    adapter_dir = experiment_dir / f"alpha_{UPLOAD_ALPHA}"
    login()
    upload_result = upload_folder(
        folder_path=str(adapter_dir),
        repo_id=HF_REPO_ID,
        repo_type="model",
        ignore_patterns=["checkpoint-*"],
        commit_message=f"Upload evaluated Qwen3.5-9B QLoRA r=48 alpha={UPLOAD_ALPHA} adapter",
    )
    print("Uploaded evaluated adapter to:", upload_result)
else:
    print("Hugging Face upload skipped. Set RUN_HF_UPLOAD=True to publish.")
